# Example: explain_certificate_file


In [ ]:
# Ensure CWD is the repo root regardless of where the notebook is launched from
# (nbmake/Jupyter set CWD to this notebook's own directory by default), and put it
# on sys.path so `from examples...` imports resolve regardless of kernel startup cwd.
import os
import sys
from pathlib import Path

_p = Path.cwd()
while not (_p / "pyproject.toml").exists() and _p != _p.parent:
    _p = _p.parent
os.chdir(_p)
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))
print("CWD:", Path.cwd())

In [ ]:
# Generate a real certificate JSON to explain (this notebook needs an existing file as --input)
import subprocess, sys, tempfile
from pathlib import Path

cert_path = Path(tempfile.gettempdir()) / "compitum_demo_certificate.json"
result = subprocess.run(
    [sys.executable, "-m", "compitum.cli", "route", "--prompt", "Sketch a proof for AM-GM inequality.", "--trace"],
    check=True, capture_output=True, text=True,
)
cert_path.write_text(result.stdout, encoding="utf-8")
print("Wrote", cert_path)

In [ ]:
from __future__ import annotations

import argparse
import json
from pathlib import Path

from examples.certificate_card import render_markdown_card  # reuse


def main() -> int:
    ap = argparse.ArgumentParser(description="Explain an existing certificate JSON as a Markdown card.")
    ap.add_argument("--input", type=Path, required=True, help="Path to certificate JSON or JSONL (uses first line)")
    args = ap.parse_args()

    p = args.input
    text = p.read_text(encoding="utf-8")
    if p.suffix.lower() == ".jsonl":
        line = text.splitlines()[0]
        data = json.loads(line)
    else:
        data = json.loads(text)
    print(render_markdown_card(data))
    return 0


import sys
sys.argv = ["explain_certificate_file.py", "--input", str(cert_path)]


if __name__ == "__main__":
    _rc = main()
    if _rc:
        raise SystemExit(_rc)

